<a href="https://colab.research.google.com/github/Indra-sekhar/FUTURE_ML_03/blob/main/FUTURE_ML_03_Resume_Screening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resume / Candidate Screening System

## Objective
Build a Machine Learning system to screen, score, and rank resumes based on job requirements.

## Use Case
Helps recruiters:
- Shortlist candidates faster
- Match skills with job roles
- Identify missing skills

In [7]:

# IMPORTS

import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# LOAD DATA

df = pd.read_csv("/content/drive/MyDrive/Documents /Resume.csv")  # change path if needed

# Keep required columns

df = df[["Resume_str", "Category"]]
df = df.dropna()

# (Optional) Reduce size for faster processing

df = df.head(500)

# TEXT CLEANING

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text

df["clean_resume"] = df["Resume_str"].apply(clean_text)

# SKILL LIST

skills_list = [
    "python", "machine learning", "data analysis",
    "html", "css", "javascript", "react",
    "java", "spring", "nlp", "deep learning",
    "sql", "excel"
]

# SKILL EXTRACTION

def extract_skills(text):
    found = []
    for skill in skills_list:
        if skill in text:
            found.append(skill)
    return found

df["skills"] = df["clean_resume"].apply(extract_skills)

# JOB DESCRIPTION

job_description = "Looking for a data scientist with python, machine learning, and data analysis skills"

job_clean = clean_text(job_description)
job_skills = extract_skills(job_clean)

print("Job Skills:", job_skills)

# TF-IDF SIMILARITY

vectorizer = TfidfVectorizer(max_features=5000)

resume_vectors = vectorizer.fit_transform(df["clean_resume"])
job_vector = vectorizer.transform([job_clean])

similarity_scores = cosine_similarity(resume_vectors, job_vector)

df["similarity_score"] = similarity_scores

# SKILL MATCH SCORE

def skill_score(resume_skills, job_skills):
    return len(set(resume_skills).intersection(set(job_skills)))

df["skill_score"] = df["skills"].apply(lambda x: skill_score(x, job_skills))

# FINAL SCORE

df["final_score"] = df["similarity_score"] * 0.7 + df["skill_score"] * 0.3

# Round scores for better display

df["final_score"] = df["final_score"].round(3)
df["similarity_score"] = df["similarity_score"].round(3)

# RANK CANDIDATES

ranked_df = df.sort_values(by="final_score", ascending=False)

ranked_df["rank"] = range(1, len(ranked_df) + 1)

print("\nTop 10 Ranked Candidates:\n")
print(ranked_df[["rank", "Category", "skills", "final_score"]].head(10))

# SKILL GAP ANALYSIS

def missing_skills(resume_skills, job_skills):
    return list(set(job_skills) - set(resume_skills))

ranked_df["missing_skills"] = ranked_df["skills"].apply(
    lambda x: missing_skills(x, job_skills)
)

print("\nTop Candidates Skill Gaps:\n")
print(ranked_df[["rank", "Category", "missing_skills"]].head(10))

# DEMO FUNCTION (VERY IMPORTANT)

def screen_resume(resume_text):
    cleaned = clean_text(resume_text)
    skills = extract_skills(cleaned)

    vector = vectorizer.transform([cleaned])
    similarity = cosine_similarity(vector, job_vector)[0][0]

    score = similarity * 0.7 + len(set(skills).intersection(set(job_skills))) * 0.3

    print("\n🔍 Resume Analysis")
    print("----------------------")
    print("Skills Found:", skills)
    print("Similarity Score:", round(similarity, 3))
    print("Final Score:", round(score, 3))
    print("Missing Skills:", list(set(job_skills) - set(skills)))

# TEST EXAMPLE

screen_resume("Python developer with machine learning and SQL experience")

Job Skills: ['python', 'machine learning', 'data analysis']

Top 10 Ranked Candidates:

     rank                Category  \
331     1  INFORMATION-TECHNOLOGY   
249     2  INFORMATION-TECHNOLOGY   
126     3                DESIGNER   
315     4  INFORMATION-TECHNOLOGY   
267     5  INFORMATION-TECHNOLOGY   
47      6                      HR   
316     7  INFORMATION-TECHNOLOGY   
471     8                ADVOCATE   
220     9  INFORMATION-TECHNOLOGY   
233    10  INFORMATION-TECHNOLOGY   

                                             skills  final_score  
331  [python, data analysis, html, css, sql, excel]        0.760  
249       [python, data analysis, javascript, java]        0.690  
126          [python, data analysis, spring, excel]        0.668  
315                      [python, java, sql, excel]        0.428  
267                            [python, sql, excel]        0.410  
47                           [data analysis, excel]        0.402  
316                [data analysis, 